# Watch Dropshipping Multi-Agent Pipeline — Stage Demo

A supplier signal comes in → 3 smolagents (ingestion, market research, copywriting) process it under a manager agent → LangGraph gates the result behind a margin-based human-approval step → an approved listing goes live on the local storefront and broadcasts to Telegram.

See `README.md` in this folder for setup. Requires `GROQ_API_KEY`; `TELEGRAM_BOT_TOKEN` / `TELEGRAM_CHAT_ID` are optional — without them, use the local fallback cell instead of the live Telegram approval.

## Cell 1 — Environment Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import os, json, time

from multi_agent_system import load_key

load_key("GROQ_API_KEY")
load_key("TELEGRAM_BOT_TOKEN") if "TELEGRAM_BOT_TOKEN" not in os.environ else None
load_key("TELEGRAM_CHAT_ID") if "TELEGRAM_CHAT_ID" not in os.environ else None

## Cell 2 — Launch the Storefront (background thread)

In [ ]:
import threading
import requests
from app import app as flask_app

threading.Thread(target=lambda: flask_app.run(port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(1)

catalog = requests.get("http://localhost:5000/api/catalog").json()
print(f"Storefront live at http://localhost:5000 — {len(catalog)} watches currently listed:")
for w in catalog:
    print(f"  - {w['title']} (${w['price']:.2f})")

## Cell 3 — Initialize the Multi-Agent Network

In [ ]:
from multi_agent_system import build_manager_agent, MANAGER_PROMPT_TEMPLATE

manager_agent = build_manager_agent()
print("Manager agent ready, managing:", list(manager_agent.managed_agents.keys()))

## Cell 4 — Simulate an Incoming Supplier Signal

In [ ]:
supplier_signal = {
    "raw_message": "HOT DROP: Skeleton Automatic Watch Matte Black. Cost $12.00/unit. Min order 5 pcs.",
    "image_path": "static/watches/skeleton_watch.jpg",
}
supplier_signal

## Cell 5 & 6 — Run the Pipeline, then LangGraph's Margin Guardrail

`run_multi_agent_node` calls the manager agent (live trace prints below), then `evaluate_margin_node` checks `margin_pct < 20%` and calls `interrupt()` if a human needs to sign off on a thin-margin listing. Margins at or above 20% auto-approve.

In [ ]:
from workflow import build_graph, resume_graph

graph = build_graph(manager_agent, MANAGER_PROMPT_TEMPLATE)
thread_config = {"configurable": {"thread_id": "dropship-demo-1"}}

initial_state = {
    "raw_message": supplier_signal["raw_message"],
    "image_path": supplier_signal["image_path"],
    "specs": {},
    "market_price": 0.0,
    "margin_pct": 0.0,
    "description": "",
    "broadcast_copy": "",
    "approved": False,
    "published": False,
}

result = graph.invoke(initial_state, thread_config)

last_decision = None
decision_channel = None

if "__interrupt__" in result:
    interrupt_payload = result["__interrupt__"][0].value
    print(">>> PAUSED — waiting for a human decision <<<")
    print(json.dumps(interrupt_payload, indent=2))
else:
    interrupt_payload = None
    last_decision = "auto-approved"
    decision_channel = "none (margin ≤ 20%)"
    print("Margin below threshold — auto-published without human review.")

## Cell 6b — Telegram HITL Trigger (primary path)

Sends an inline-keyboard approval message to your phone and starts a background listener. Skip this cell (and use the local fallback below instead) if `TELEGRAM_BOT_TOKEN` / `TELEGRAM_CHAT_ID` aren't set.

In [ ]:
from telegram_bot import send_approval_request, start_approval_listener

resumed_state = {}


def on_decision(decision: str, channel: str = "Telegram"):
    global last_decision, decision_channel
    last_decision = decision
    decision_channel = channel
    final = resume_graph(graph, thread_config, decision)
    resumed_state.update(final)
    print(f"[workflow] resumed with decision={decision!r} via {channel} -> published={final.get('published')}")


if interrupt_payload and os.environ.get("TELEGRAM_BOT_TOKEN") and os.environ.get("TELEGRAM_CHAT_ID"):
    bot_app = start_approval_listener(os.environ["TELEGRAM_BOT_TOKEN"], on_decision)
    await send_approval_request(os.environ["TELEGRAM_BOT_TOKEN"], os.environ["TELEGRAM_CHAT_ID"], interrupt_payload)
    print("Approval request sent — tap a button on your phone now.")
else:
    print("Telegram not configured (or nothing to approve) — use the local fallback cell instead.")

## Cell 7 — Local Fallback (only if Telegram is unreachable)

Same resume path as the Telegram callback — run this instead of waiting on a phone tap.

In [ ]:
if interrupt_payload and not resumed_state:
    decision = input("Approve publishing? (approve/reject): ").strip().lower()
    on_decision(decision, channel="Local fallback")

## Cell 8 — Verify the Website Update + Broadcast the Deal

In [ ]:
final_state = resumed_state or result

if final_state.get("published"):
    catalog = requests.get("http://localhost:5000/api/catalog").json()
    print("Top of catalog now:")
    print(json.dumps(catalog[0], indent=2))

    if os.environ.get("TELEGRAM_BOT_TOKEN") and os.environ.get("TELEGRAM_CHAT_ID"):
        from telegram_bot import broadcast_deal
        await broadcast_deal(os.environ["TELEGRAM_BOT_TOKEN"], os.environ["TELEGRAM_CHAT_ID"], final_state["broadcast_copy"])
        print("Broadcast sent to Telegram.")
else:
    print("Not published (rejected or still pending).")

## Cell 9 — Run Recap

In [ ]:
from IPython.display import Markdown, display

specs = final_state.get("specs", {})

recap_rows = [
    ("Supplier message", supplier_signal["raw_message"]),
    ("Supplier image", supplier_signal["image_path"]),
    ("Extracted title", specs.get("title", "—")),
    ("Supplier cost", f"${specs.get('base_cost', 0):.2f}" if specs.get("base_cost") is not None else "—"),
    ("Market price found", f"${final_state.get('market_price', 0):.2f}"),
    ("Margin", f"{final_state.get('margin_pct', 0):.1f}%"),
    ("Approval needed?", "Yes (margin < 20%)" if interrupt_payload else "No (auto-approved, margin >= 20%)"),
    ("Decision", f"{last_decision or '—'} via {decision_channel or '—'}"),
    ("Published to storefront?", "Yes" if final_state.get("published") else "No"),
    ("Broadcast sent?", "Yes" if final_state.get("published") and os.environ.get("TELEGRAM_BOT_TOKEN") else "No"),
]

table_md = "| Step | Result |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in recap_rows)
display(Markdown(table_md))